# KosenMap Website 取扱説明書 —— 目次

**この Website(公開ページ・管理画面・本番ホスト)を動かすための取扱説明書。**
読むだけでなく、**書いてあるコマンドをそのセルから実行できる。**

対象は本番 `ito4.jp`(`/opt/kosenmap`)と、この PC の `server/scripts`。
**繋ぐ利用者は 2 つ**(2026-09-18 に分けた。[12](12-hardening-2026-09-15.ipynb) §7-4 B): 配備・`%%host`・控えは **`kmops`**(鍵 `~\.ssh\km_ops`。docker あり・**sudo なし**)、`sudo` が要るセルは **`km`**(鍵 `~\.ssh\km_vps`。**docker なし**)。
Android アプリは別リポジトリなので、ここには入っていない。

| ノートブック | 何が書いてあるか |
|---|---|
| [00-start](00-start.ipynb) | **この取扱説明書の使い方。** 準備と、セルの型 |
| [01-daily-check](01-daily-check.ipynb) | **日々の確認。** 自己検査・ホストの様子・メールが届いているか |
| [02-deploy](02-deploy.ipynb) | **配備。** 下見・配備・後片付け・困ったとき |
| [03-backup](03-backup.ipynb) | **バックアップと復元。** 取る・開く・週次タスク・添付の復号・戻す |
| [04-host-jobs-and-mail](04-host-jobs-and-mail.ipynb) | **ホストの定期処理とメール。** cron の時刻表・届くメールの一覧・試しに送る |
| [05-containers](05-containers.ipynb) | **コンテナの更新と追加。** Logto の上げ方・固定・足し方 |
| [06-emergency](06-emergency.ipynb) | **もしものとき。** まず叩く1本と、症状別の見どころ |
| [07-map-qr](07-map-qr.ipynb) | **地図の配信と QR。** |
| [08-architecture](08-architecture.ipynb) | **どう出来ているか。** 構造・設定の読み方・落とし穴・設計の約束 |
| [09-new-host](09-new-host.ipynb) | **新しいホストを作る・移す。** VPS の構築から切り替えまで |
| [10-security-review-2026-09-14](10-security-review-2026-09-14.ipynb) | **診断の報告書(2026-09-14)。** 何を見つけて何を直したか・本番で実行する手順 |
| [11-getting-started](11-getting-started.ipynb) | **新しく使う人の入口。** 全体の図・部品の役割・管理画面・Android アプリ・用語集 |
| [12-hardening-2026-09-15](12-hardening-2026-09-15.ipynb) | **多層防御の底上げ(2026-09-15)。** 網の分割・read_only・Soketi の Node 24・Logto の DB 利用者・管理画面の別オリジン・本番への当て方 |
| [13-local-env](13-local-env.ipynb) | **ローカル環境(LAN の検証機)。** `.env` の `KM_ENV=local` で、Let's Encrypt を使わずに本番と同じ構成を立てる |
| [14-domain-ito4](14-domain-ito4.ipynb) | **ito4.jp に統一する(2026-09-17)。** 旧 ito8795.com は同時に手放す・audience も https://ito4.jp/api へ・管理画面は admin.ito4.jp・外部スキャンの指摘 |
| [15-staff-org](15-staff-org.ipynb) | **教職員の自動付与(2026-09-18〜)。** 学校ドメインの JIT で Logto の組織へ入れ、組織ロールで「地図の錠を通る・教職員氏名を見る・閲覧不可の地点を見る」を与える |

記録として残している文書:

| 文書 | 中身 |
|---|---|
| [status.md](status.md) | **いまどうなっているか。** 実測値・踏んだ罠・残作業 |
| [plan.md](plan.md) | **これから何をするか。** 優先順と、やらないと決めたこと |
| [../Old/docs/](../Old/docs/) | 以前の手順書(md)。**細部の経緯はここ**。ノートブックに移した内容の元 |

# 地図の配信と QR

**Android アプリへの地図の配信は、管理画面から行う。** 手元のスクリプトから本番へ書き込む方式は 2026-09-03 にやめた。

> 使い方は [00-start.ipynb](00-start.ipynb)。**まず下のセルを1回実行する。**

| 印 | 意味 |
|---|---|
| 🟢 | **読むだけ。** 何も変えない。迷ったらここから |
| 🟡 | **手元が変わる。** この PC にファイルを作る・登録する。本番には触れない |
| 🔴 | **本番が変わる。** 実行前に `yes` の入力を求める |
| 🔑 | **別の窓で開く。** sudo のパスワードなど対話が要るもの |

In [ ]:
# 最初に1回だけ実行する(%%ps / %%host / %%terminal が使えるようになる)
import sys, pathlib
for _d in (pathlib.Path.cwd(), pathlib.Path.cwd() / 'docs', pathlib.Path.cwd() / 'server' / 'docs'):
    if (_d / 'km_nb.py').exists():
        sys.path.insert(0, str(_d))
        break
import km_nb
km_nb.load()

## 1. 配信(管理画面)

`https://admin.ito4.jp/admin/map-publish.php`

- 配る JSON(`uploads/app-map-<slug>.json`)は**生成物**。正本は DB。**上げた JSON をそのまま配らない**
- **有効期限は必ず入れる。** 空のまま配ると期限無しになり、端末に残った地図がいつまでも消えない
- アクセスコードは**作った直後だけ画面に出る**(URL には載せない)。控え損ねたら作り直す
- 画面の QR は `src/lib/qr.php`(外部のサービスは使わない)

> 以前の `Old/new-map-release.ps1` と、それが呼ぶ `scripts/app-map-config-set.php` は、もう日常では使わない。
> `app-map-config-set.php` はアクセスコードを**標準入力で**受けてハッシュ化する(`ps` に平文を出さないため)。

## 2. QR を手元で作る(new-map-qr.ps1)

会場に貼るなど、画面の外で使う QR。**誤り訂正は `Q` か `H`**(汚れや一部が隠れても読める)を勧める。
コードにはプレフィックスが自動で付くので、**コードだけ**を渡す。

### 画面に出して確かめる

`TEST123` を実際のコードに書き換える。ファイルは作らない。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%ps
.\new-map-qr.ps1 -Code TEST123 -ErrorCorrection Q -MatrixOnly | Select-Object -First 3

### PNG を作る

ダウンロードフォルダに作る。

🟡 **手元が変わる** —— この PC にファイルを作る・消す、登録するなど。本番には触れません。

In [ ]:
%%ps
$code = 'KOSEN2026'   # ← 実際のコード
.\new-map-qr.ps1 -Code $code -ErrorCorrection Q -ModuleSize 12 -OutputPath "$env:USERPROFILE\Downloads\kosenmap-qr-$code.png"

### 3. PHP 版と PowerShell 版が一致するか

管理画面の QR(PHP)と、手元の QR(PowerShell)が同じ並びを作るかを突き合わせる。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%ps
.\check-qr-port.ps1

### 4. 地図まわりの自己検査

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%ps
php ..\src\scripts\check.php qr app-map app-map-convert map-access map-events